In [ ]:
import os 
import threading
import time
from multiprocessing import Process, Queue
import traceback

path = "C:/Users/lucas/Documents/code"
# path = "test_arbo"
search = "tqsbbxvn.txt".lower()
start = os.listdir(path)
stacks = []
threads = []
found = []

def unstack(liste:Queue, trace):
    n = liste.qsize()

    while liste.qsize() != 0:
        path_el = liste.get()
        path_full = f"{path}/{path_el}"

        try:
            l = os.listdir(path_full)
            n += len(l)
            for el in l:
                liste.put(f"{path_el}/{el}")
        except:
            if path_el.split("/")[-1].lower() == search:
                found.append(path_full)
    
        # del liste[0]

    print(n)
    return found

def initStack(nbStacks):
    for i in range(nbStacks):
        stacks.append(Queue())
    for j, el in enumerate(start):
        stacks[j%nbStacks].put(el)

    for i in range(nbStacks):
        t = Process(target=unstack, args=(stacks[i], i))
        threads.append(t)
    return stacks



nbStacks = 3
initStack(nbStacks)

ti = time.time()
for t in threads:
    t.start()
for t in threads:
    t.join()
unstack(stacks[1], 1)
print(time.time() - ti)
# print(threads)
# print(found)
# print(traceback.print_exception(None, threads[0], threads[0].__traceback__))

# threads[0].start()
# time.sleep(5)
# threads[0]




KeyboardInterrupt: 

In [13]:
import os 
import threading
import time
import numpy as np

path = "C:/Users/lucas/Documents/code"
search = "errorFatal.ejs".lower()
start = os.listdir(path)

def unstack(i):

    while len(stacks[i]) != 0:
        path_el = stacks[i][0]
        stacks[i].remove(path_el)
        
        path_full = f"{path}/{path_el}"
        
        if os.path.isfile(path_full):
            if path_el.split("/")[-1].lower() == search:
                found.append(path_full)
        else:
            l = os.listdir(path_full)
            l = list(map(lambda x:f"{path_el}/{x}", l))
            stacks[i] += l
        
        if len(stacks[i]) == 0:
            lens = list(map(lambda x:len(x), stacks))
            j = lens.index(max(lens))
            stacks[j], stacks[i] = stacks[j][:len(stacks[j])//2], stacks[j][len(stacks[j])//2:]

        # print(trace)
    
    return found

def initStack(nbStacks):
    for i in range(nbStacks):
        stacks.append([])
    for j, el in enumerate(start):
        stacks[j%nbStacks].append(el)

    for i in range(nbStacks):
        t = threading.Thread(target=unstack, args=(i, ))
        threads.append(t)
    return stacks

d = {}
for nbStacks in range(2, 20):
    print(nbStacks)
    l = []
    for i in range(3):

        stacks = []
        threads = []
        found = []
        initStack(nbStacks)

        td = time.time()
        

        # print(list(map(lambda x:len(x), stacks)))
        for t in threads:
            t.start()
        for t in threads:
            t.join()

        l.append(time.time() - td)
        # print(stacks)
    d[nbStacks] = np.mean(l)

# alive = True
# while sum(map(lambda x:x.is_alive(), threads)) != 0:
#     for i, t in enumerate(threads):
#         if not t.is_alive():
#             print(i)
#             stacks[i-1], split_stack = stacks[i-1][:len(stacks[i-1])//2], stacks[i-1][len(stacks[i-1])//2:]
#             t_new = threading.Thread(target=unstack, args=(stacks[i], i))
#             threads.remove(t)

#             t_new.start()
#             threads.append(t_new)


    

d

2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19


{2: np.float64(2.0990055402119956),
 3: np.float64(1.67915145556132),
 4: np.float64(1.567650318145752),
 5: np.float64(1.669169267018636),
 6: np.float64(1.7513261636098225),
 7: np.float64(2.0222861766815186),
 8: np.float64(2.217916170756022),
 9: np.float64(2.331177075703939),
 10: np.float64(2.829897880554199),
 11: np.float64(3.043776591618856),
 12: np.float64(2.9092787901560464),
 13: np.float64(2.7448483308156333),
 14: np.float64(2.908443053563436),
 15: np.float64(2.9944852193196616),
 16: np.float64(2.9120771884918213),
 17: np.float64(3.001306931177775),
 18: np.float64(2.917454719543457),
 19: np.float64(3.0554562409718833)}

In [17]:
sum(map(lambda x:x.is_alive(), threads))

0

In [5]:
import os
from multiprocessing import Process, Manager, Queue

def worker(i, path_root, search, stack, queues, found):
    """Worker avec pile locale + work-stealing."""
    nb_workers = len(queues)
    queue = queues[i]
    while len(stack) != 0:

        rel_path = stack.pop()
        full_path = os.path.join(path_root, rel_path)

        if os.path.basename(full_path).lower() == search:
            found.append(full_path)
        elif not os.path.isfile(full_path):
            try:
                children = os.listdir(full_path)
                stack.extend(list(map(lambda x:os.path.join(rel_path, x), children)))
            except PermissionError:
                continue

        if len(stack) == 0:
            print(i, "help")

            if queue.qsize() != 0:
                for j in range(nb_workers):
                    if j != i:
                        queues[j].put(("need", i))

            try:
                msg, content = queue.get(timeout=2)
            except:
                # personne n’a plus rien → fin
                continue

            if msg == "donate":
                stack.extend(content)
        
        if queue.qsize() != 0:
            try:
                msg, content = queue.get(timeout=0.5)
                if msg == "need":
                    print(i, "don")
                    half = len(stack) // 2
                    queues[content].put(("donate", stack[half:]))
                    stack = stack[:half]
            except:
                continue



def parallel_search(path, search, nb_workers=3):
    manager = Manager()

    # liste de piles partagée → chaque pile est une Manager.list()
    stacks = [[] for _ in range(nb_workers)]
    found = manager.list()
    queues = [Queue() for _ in range(nb_workers)]

    # distribution initiale du travail comme ton initStack()
    start = os.listdir(path)
    for j, el in enumerate(start):
        stacks[j % nb_workers].append(el)

    processes = []
    for i in range(nb_workers):
        p = Process(target=worker, args=(i, path, search, stacks[i], queues, found))
        # worker(i, path, search, stacks[i], queues, found)
        p.start()
        processes.append(p)

    for p in processes:
        p.join()

    return list(found)


# ===============================
# Exemple d'utilisation
# ===============================
if __name__ == "__main__":
    path = "C:/Users/lucas/Documents/code"
    search = "errorFatal.ejs".lower()

    results = parallel_search(path, search, nb_workers=3)

    print("Found:")
    for r in results:
        print(r)


Found:
